#### PyTorch Fundamentals

Core data structure of PyTorch is a tensor. A multidimensional array with a shape and a data type. Kind of like an array, but with support for GPU and support auto-differentiation.

Creation of a tensor

In [3]:
import torch

# let's create a 2 x 3 array (tensor)
X = torch.tensor([[1, 2, 3], [4, 5, 6]])
X

tensor([[1, 2, 3],
        [4, 5, 6]])

Info

In [4]:
X.shape

torch.Size([2, 3])

In [5]:
X.dtype

torch.int64

Indexing

In [6]:
X[0, 1]

tensor(2)

In [7]:
X[:, 1]

tensor([2, 5])

#### Hardware Acceleration

In [8]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

Now let's create a tensor on that GPU

In [9]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M = M.to(device)

In [10]:
M.device

device(type='cuda', index=0)

In [11]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

Run something on the gpu

In [12]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

In [13]:
M = torch.rand((1000, 1000))  # on the CPU
%timeit M @ M.T

6.11 ms ± 179 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
M = torch.rand((1000, 1000), device="cuda")  # on the GPU
%timeit M @ M.T

228 μs ± 5.22 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


#### Autograd

PyTorch comes with an efficient implementation of reverse-mode auto-differentiation, called `autograd`, which stands for `automatic gradients`.

Consider a simple function, f(x) = x^2. Derivitive of this is f'(x) = 2x so eval on f(x) and f'(x) should be 25 and 10.

In [15]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [16]:
f.backward()
x.grad

tensor(10.)

After computing the gradients, you generally want to perform a gradien descent step by subtracting a fraction of the gradients from the model variables.

In [17]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

or use `detach()` which is detached from the computational graph.

In [18]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

Before repeating the process (forward pass + backward pass + gradient descent step), it is essential to zero out the gradients of every model params.

In [19]:
x.grad.zero_()

tensor(0.)

Putting it all together, this how the training loop looks like:

In [20]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2 # forward pass
    f.backward() # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad # gradient descent step

    x.grad.zero_()

#### Implementing Linear Regression

In [21]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

Let's cast our dataset to FloatTensor

In [22]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

And convert our targets as well

In [23]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

Let's create the params for our model

In [24]:
torch.manual_seed(42)
n_features = X_train.shape[1] # there are 8 input features
w = torch.rand((n_features, 1), requires_grad=True) # weights
b = torch.tensor(0., requires_grad=True) # bias

Next, let's train our model.

In [25]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 7.792580604553223
Epoch 2/20, Loss: 1.3588199615478516
Epoch 3/20, Loss: 0.84256911277771
Epoch 4/20, Loss: 0.7366544604301453
Epoch 5/20, Loss: 0.6970381140708923
Epoch 6/20, Loss: 0.6746658086776733
Epoch 7/20, Loss: 0.6581473350524902
Epoch 8/20, Loss: 0.6443582773208618
Epoch 9/20, Loss: 0.6323144435882568
Epoch 10/20, Loss: 0.6216264367103577
Epoch 11/20, Loss: 0.6120853424072266
Epoch 12/20, Loss: 0.6035464406013489
Epoch 13/20, Loss: 0.5958940982818604
Epoch 14/20, Loss: 0.5890297889709473
Epoch 15/20, Loss: 0.5828677415847778
Epoch 16/20, Loss: 0.577332079410553
Epoch 17/20, Loss: 0.5723556280136108
Epoch 18/20, Loss: 0.5678789019584656
Epoch 19/20, Loss: 0.5638489723205566
Epoch 20/20, Loss: 0.5602186918258667


Now we can predict

In [26]:
X_new = X_test[:3] # pretend these are new instances
with torch.no_grad():
    y_pred = X_new @ w + b # use the trained params to make predictions
y_pred

tensor([[0.8684],
        [1.6626],
        [2.7272]])

Now let's rewrite using higher-level API.

In [27]:
import torch.nn as nn  # by convention, this module is usually imported this way
torch.manual_seed(42)  # to get reproducible results
model = nn.Linear(in_features=n_features, out_features=1)

In [28]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [29]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

We can call the model as a function

In [30]:
model(X_train[:2])

tensor([[-0.4718],
        [ 0.1131]], grad_fn=<AddmmBackward0>)

We have our model, now we need to create an optimizer to update the model params and must also choose a loss function

In [31]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

Let's define a simple train function. Notice it is simpler than before.

In [32]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [33]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.3378496170043945
Epoch 2/20, Loss: 0.780293345451355
Epoch 3/20, Loss: 0.6253840923309326
Epoch 4/20, Loss: 0.6060433387756348
Epoch 5/20, Loss: 0.5956299304962158
Epoch 6/20, Loss: 0.5873566269874573
Epoch 7/20, Loss: 0.5802990198135376
Epoch 8/20, Loss: 0.5741382241249084
Epoch 9/20, Loss: 0.5687100887298584
Epoch 10/20, Loss: 0.5639079213142395
Epoch 11/20, Loss: 0.5596510767936707
Epoch 12/20, Loss: 0.5558737516403198
Epoch 13/20, Loss: 0.5525194406509399
Epoch 14/20, Loss: 0.5495391488075256
Epoch 15/20, Loss: 0.5468900203704834
Epoch 16/20, Loss: 0.544533908367157
Epoch 17/20, Loss: 0.5424376726150513
Epoch 18/20, Loss: 0.5405716300010681
Epoch 19/20, Loss: 0.5389097332954407
Epoch 20/20, Loss: 0.5374288558959961


In [34]:
X_new = X_test[:3]  # pretend these are new instances
with torch.no_grad():
    y_pred = model(X_new)  # use the trained model to make predictions
y_pred

tensor([[0.8061],
        [1.7116],
        [2.6973]])

#### Implementing a Regression MLP

The MLP is a simple stack of modules - two hidden layers and one output layer.

In [35]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [36]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 5.045480251312256
Epoch 2/20, Loss: 2.0523128509521484
Epoch 3/20, Loss: 1.0039883852005005
Epoch 4/20, Loss: 0.8570139408111572
Epoch 5/20, Loss: 0.7740675210952759
Epoch 6/20, Loss: 0.7225848436355591
Epoch 7/20, Loss: 0.6893726587295532
Epoch 8/20, Loss: 0.6669032573699951
Epoch 9/20, Loss: 0.6507739424705505
Epoch 10/20, Loss: 0.6383935213088989
Epoch 11/20, Loss: 0.6281994581222534
Epoch 12/20, Loss: 0.6193399429321289
Epoch 13/20, Loss: 0.6113173365592957
Epoch 14/20, Loss: 0.6038705706596375
Epoch 15/20, Loss: 0.5968308448791504
Epoch 16/20, Loss: 0.5901119112968445
Epoch 17/20, Loss: 0.5836468935012817
Epoch 18/20, Loss: 0.5774064064025879
Epoch 19/20, Loss: 0.5713555812835693
Epoch 20/20, Loss: 0.565444827079773


we've trained a neural net.

#### Implementing Mini-Batch Gradient Descent Using DataLoaders

In [37]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)

We have a larger model and we have the tools to train it batch at a time. Let's use the GPU.

In [38]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)
model = model.to(device, non_blocking=True)

Write a generic train loop

In [39]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [40]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 5.0460
Epoch 2/20, Loss: 5.0443
Epoch 3/20, Loss: 5.0456
Epoch 4/20, Loss: 5.0457
Epoch 5/20, Loss: 5.0450
Epoch 6/20, Loss: 5.0452
Epoch 7/20, Loss: 5.0449
Epoch 8/20, Loss: 5.0449
Epoch 9/20, Loss: 5.0459
Epoch 10/20, Loss: 5.0460
Epoch 11/20, Loss: 5.0456
Epoch 12/20, Loss: 5.0449
Epoch 13/20, Loss: 5.0461
Epoch 14/20, Loss: 5.0452
Epoch 15/20, Loss: 5.0456
Epoch 16/20, Loss: 5.0454
Epoch 17/20, Loss: 5.0455
Epoch 18/20, Loss: 5.0460
Epoch 19/20, Loss: 5.0451
Epoch 20/20, Loss: 5.0456


#### Model Evaluation

We will write an eval function.

In [41]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

Now let's build a `TensorDataset` and a `DataLoader` for our validation set, and pass it to our `evaluate()` function to compute the validation MSE.

In [42]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(4.8635, device='cuda:0')

If we want to use RMSE instead of MSE, there are built-ins but it's easy enough to write.

In [43]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

evaluate(model, valid_loader, rmse)

tensor(2.1945, device='cuda:0')

In [44]:
valid_mse.sqrt()

tensor(2.2053, device='cuda:0')

the RMSE should be equal to the square root of the MSE, however, we got a diff. result

We calculated the RMSE over each batch and then computed the mean for all these batches. That's not equivalent to computing the RMSE over the whole validation set.

In [45]:
evaluate(model,
         valid_loader,
         mse,
         aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics))
         )

tensor(2.2053, device='cuda:0')